# Summarization Model — TextRank Pipeline

**Extractive summarization** using the TextRank algorithm for chat conversation summaries.

**What this notebook does:**
1. Implements the TextRank algorithm (matches `ai/summarization/model/textrank.py`)
2. Builds a cosine-similarity sentence graph with PageRank scoring
3. Tests on sample chat conversations
4. Tunes hyperparameters (damping factor, num_sentences)
5. Evaluates with ROUGE-like metrics
6. Demonstrates the full summarization pipeline

**Note:** TextRank is an **extractive, unsupervised** algorithm — it does **not** require training data or GPU.  
It selects the most important sentences from the input using graph-based ranking.

In [ ]:
# ============================================================
# Cell 1 — Install dependencies (run once)
# ============================================================
!pip install -q numpy matplotlib

In [ ]:
# ============================================================
# Cell 2 — Imports
# ============================================================
import numpy as np
import re
from collections import Counter
import matplotlib.pyplot as plt

## 1. TextRank Summarizer

Matches `ai/summarization/model/textrank.py` exactly.  
Uses cosine similarity on bag-of-words vectors + PageRank to rank sentences.

In [ ]:
# ============================================================
# Cell 3 — TextRank Summarizer (matches ai/summarization/model/textrank.py)
# ============================================================
class TextRankSummarizer:
    """Extractive summarization using TextRank algorithm."""

    def __init__(self, damping=0.85, min_diff=1e-5, max_iter=100):
        self.damping = damping
        self.min_diff = min_diff
        self.max_iter = max_iter

    def summarize(self, sentences, num_sentences=5):
        if len(sentences) <= num_sentences:
            return " ".join(sentences)

        sim_matrix = self._build_similarity_matrix(sentences)
        scores = self._pagerank(sim_matrix)

        ranked_indices = np.argsort(scores)[::-1][:num_sentences]
        ranked_indices = sorted(ranked_indices)

        summary_sentences = [sentences[i] for i in ranked_indices]
        return " ".join(summary_sentences)

    def get_sentence_scores(self, sentences):
        """Return PageRank scores for each sentence (for analysis)."""
        if len(sentences) <= 1:
            return np.ones(len(sentences))
        sim_matrix = self._build_similarity_matrix(sentences)
        return self._pagerank(sim_matrix)

    def _build_similarity_matrix(self, sentences):
        n = len(sentences)
        sim_matrix = np.zeros((n, n))
        tokenized = [self._tokenize(s) for s in sentences]

        for i in range(n):
            for j in range(n):
                if i != j:
                    sim_matrix[i][j] = self._cosine_similarity(tokenized[i], tokenized[j])

        row_sums = sim_matrix.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        sim_matrix = sim_matrix / row_sums
        return sim_matrix

    def _cosine_similarity(self, tokens_a, tokens_b):
        all_tokens = list(set(tokens_a + tokens_b))
        if not all_tokens:
            return 0.0

        vec_a = np.array([tokens_a.count(t) for t in all_tokens], dtype=float)
        vec_b = np.array([tokens_b.count(t) for t in all_tokens], dtype=float)

        dot = np.dot(vec_a, vec_b)
        norm_a = np.linalg.norm(vec_a)
        norm_b = np.linalg.norm(vec_b)

        if norm_a == 0 or norm_b == 0:
            return 0.0
        return dot / (norm_a * norm_b)

    def _pagerank(self, matrix):
        n = matrix.shape[0]
        scores = np.ones(n) / n

        for iteration in range(self.max_iter):
            new_scores = (1 - self.damping) / n + self.damping * matrix.T @ scores
            if np.abs(new_scores - scores).sum() < self.min_diff:
                break
            scores = new_scores

        return scores

    @staticmethod
    def _tokenize(text):
        text = text.lower()
        text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
        tokens = text.split()
        stop_words = {"the", "a", "an", "is", "was", "are", "were", "be", "been", "being",
                      "have", "has", "had", "do", "does", "did", "will", "would", "could",
                      "should", "may", "might", "shall", "can", "to", "of", "in", "for",
                      "on", "with", "at", "by", "from", "it", "this", "that", "i", "you",
                      "he", "she", "we", "they", "me", "him", "her", "us", "them", "my",
                      "your", "his", "its", "our", "their", "and", "but", "or", "not", "so"}
        return [t for t in tokens if t not in stop_words and len(t) > 1]

summarizer = TextRankSummarizer()
print("TextRankSummarizer initialized.")
print(f"  Damping factor: {summarizer.damping}")
print(f"  Max iterations: {summarizer.max_iter}")
print(f"  Convergence threshold: {summarizer.min_diff}")

## 2. Sample Chat Conversations

We create realistic multi-message chat conversations to test the summarizer.

In [ ]:
# ============================================================
# Cell 4 — Sample chat conversations for testing
# ============================================================
conversations = {
    "Project Planning": [
        "We need to finalize the project timeline by Friday.",
        "The backend API is almost done, just need to add authentication.",
        "I think we should use JWT tokens for auth.",
        "Good idea. The frontend team is waiting for the API docs.",
        "Can we set up a meeting tomorrow to discuss the deployment plan?",
        "Sure, let me check my calendar.",
        "Also we need to write unit tests before the release.",
        "The QA team already started writing integration tests.",
        "Great, so we are on track for the beta launch next week.",
        "I will update the project board with the latest status.",
        "Don't forget to update the README with the new setup instructions.",
        "Will do. Let me know if anything else needs attention.",
    ],
    "Bug Discussion": [
        "There's a critical bug in the payment processing module.",
        "Users are reporting duplicate charges on their credit cards.",
        "I looked into it, seems like the retry logic is firing twice.",
        "We need to add an idempotency key to prevent duplicate transactions.",
        "I can implement that today. Should take about 3 hours.",
        "Also check if the webhook handler has the same issue.",
        "Good point, I will review the webhook code as well.",
        "We should add a database constraint to prevent duplicate entries.",
        "Let me create a hotfix branch for this.",
        "Make sure to add regression tests covering this scenario.",
    ],
    "Team Standup": [
        "Good morning everyone, let's start the standup.",
        "Yesterday I completed the user profile page redesign.",
        "Today I will work on the notification system.",
        "I finished the database migration scripts yesterday.",
        "I'm blocked on the third-party API integration, still waiting for credentials.",
        "I will follow up with the vendor about those credentials today.",
        "I started working on the search feature using elasticsearch.",
        "The performance tests showed a 40% improvement after the cache update.",
        "Nice work! Let's discuss the search feature in detail after standup.",
        "Sounds good. I also need to review the PR for the settings page.",
    ],
    "Casual Chat": [
        "Hey did you watch the game last night?",
        "Yeah it was amazing, the last quarter was intense.",
        "I can't believe they won in overtime.",
        "The new player really showed up when it mattered.",
        "We should organize a watch party for the finals.",
        "Great idea, my place or the office lounge?",
        "Office lounge has the big screen, let's do that.",
        "I'll bring snacks and drinks.",
        "Perfect, I'll send out invites to the team.",
    ],
}

print(f"Loaded {len(conversations)} sample conversations:")
for name, msgs in conversations.items():
    print(f"  {name}: {len(msgs)} messages")

## 3. Run Summarization on Conversations

In [ ]:
# ============================================================
# Cell 5 — Summarize each conversation
# ============================================================
for name, messages in conversations.items():
    print("=" * 70)
    print(f"Conversation: {name} ({len(messages)} messages)")
    print("=" * 70)

    # Filter very short messages (matches SummarizationPredictor logic)
    meaningful = [m for m in messages if len(m.split()) > 2]

    for n_sent in [3, 5]:
        summary = summarizer.summarize(meaningful, num_sentences=n_sent)
        print(f"\n  Summary ({n_sent} sentences):")
        for sent in summary.split(". "):
            s = sent.strip()
            if s:
                print(f"    • {s}" if not s.endswith(".") else f"    • {s}")
    print()

## 4. Analyze Sentence Scores

Visualize how PageRank assigns importance scores to each sentence.

In [ ]:
# ============================================================
# Cell 6 — Visualize sentence importance scores
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, (name, messages) in zip(axes.flatten(), conversations.items()):
    meaningful = [m for m in messages if len(m.split()) > 2]
    scores = summarizer.get_sentence_scores(meaningful)

    # Truncate labels for readability
    labels = [m[:40] + "..." if len(m) > 40 else m for m in meaningful]

    colors = ["#2196F3" if s >= np.sort(scores)[-3] else "#BBDEFB" for s in scores]

    y_pos = np.arange(len(labels))
    ax.barh(y_pos, scores, color=colors)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel("PageRank Score")
    ax.set_title(f"{name}", fontsize=11, fontweight="bold")
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis="x")

plt.suptitle("TextRank — Sentence Importance Scores", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Hyperparameter Tuning — Damping Factor

The damping factor controls how much the algorithm favors highly-connected sentences.  
Standard value is 0.85 (same as Google's original PageRank).

In [ ]:
# ============================================================
# Cell 7 — Damping factor comparison
# ============================================================
test_conv = conversations["Project Planning"]
meaningful = [m for m in test_conv if len(m.split()) > 2]

damping_values = [0.5, 0.7, 0.85, 0.95]

print("Effect of Damping Factor on Summary")
print("=" * 70)

for d in damping_values:
    s = TextRankSummarizer(damping=d)
    summary = s.summarize(meaningful, num_sentences=3)
    print(f"\n  Damping = {d}:")
    for sent in summary.split(". "):
        sent = sent.strip()
        if sent:
            print(f"    • {sent}")

# Visualize score distribution for different damping values
fig, axes = plt.subplots(1, len(damping_values), figsize=(20, 5), sharey=True)

for ax, d in zip(axes, damping_values):
    s = TextRankSummarizer(damping=d)
    scores = s.get_sentence_scores(meaningful)
    ax.bar(range(len(scores)), scores, color="#42A5F5")
    ax.set_title(f"Damping = {d}")
    ax.set_xlabel("Sentence Index")
    ax.set_ylabel("Score")
    ax.grid(True, alpha=0.3)

plt.suptitle("Score Distribution by Damping Factor", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. ROUGE-like Evaluation

Since TextRank is unsupervised, we evaluate using overlap metrics between the summary and the full conversation.  
We compute **coverage** (how much of the original content is represented) and **compression ratio**.

In [ ]:
# ============================================================
# Cell 8 — Evaluation metrics
# ============================================================
def compute_word_overlap(summary, original_sentences):
    """Compute word-level overlap between summary and original."""
    def tokenize(text):
        return set(re.sub(r"[^a-zA-Z0-9\s]", "", text.lower()).split())

    summary_words = tokenize(summary)
    original_words = set()
    for sent in original_sentences:
        original_words.update(tokenize(sent))

    if not original_words:
        return 0.0
    overlap = summary_words & original_words
    coverage = len(overlap) / len(original_words)
    return coverage

def compute_compression_ratio(summary, original_sentences):
    """Ratio of summary length to original length."""
    orig_len = sum(len(s.split()) for s in original_sentences)
    summ_len = len(summary.split())
    return summ_len / max(orig_len, 1)

def compute_bigram_overlap(summary, original_sentences):
    """ROUGE-2-like: bigram overlap."""
    def get_bigrams(text):
        words = re.sub(r"[^a-zA-Z0-9\s]", "", text.lower()).split()
        return set(tuple(words[i:i+2]) for i in range(len(words) - 1))

    summary_bigrams = get_bigrams(summary)
    original_bigrams = set()
    for sent in original_sentences:
        original_bigrams.update(get_bigrams(sent))

    if not original_bigrams:
        return 0.0
    overlap = summary_bigrams & original_bigrams
    return len(overlap) / len(original_bigrams)


print("=" * 70)
print("Evaluation Metrics")
print("=" * 70)

results = []
for name, messages in conversations.items():
    meaningful = [m for m in messages if len(m.split()) > 2]
    for n_sent in [3, 5]:
        summary = summarizer.summarize(meaningful, num_sentences=n_sent)
        coverage = compute_word_overlap(summary, meaningful)
        compression = compute_compression_ratio(summary, meaningful)
        bigram_ovlp = compute_bigram_overlap(summary, meaningful)

        results.append({
            "conversation": name,
            "num_sentences": n_sent,
            "word_coverage": coverage,
            "bigram_coverage": bigram_ovlp,
            "compression_ratio": compression,
        })

        print(f"\n  {name} (top {n_sent} sentences):")
        print(f"    Word Coverage:    {coverage:.2%}")
        print(f"    Bigram Coverage:  {bigram_ovlp:.2%}")
        print(f"    Compression:      {compression:.2%}")

In [ ]:
# ============================================================
# Cell 9 — Visualize evaluation results
# ============================================================
conv_names = list(conversations.keys())
x = np.arange(len(conv_names))
width = 0.35

# Coverage for 3 vs 5 sentences
cov_3 = [r["word_coverage"] for r in results if r["num_sentences"] == 3]
cov_5 = [r["word_coverage"] for r in results if r["num_sentences"] == 5]
comp_3 = [r["compression_ratio"] for r in results if r["num_sentences"] == 3]
comp_5 = [r["compression_ratio"] for r in results if r["num_sentences"] == 5]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(x - width/2, cov_3, width, label="3 sentences", color="#42A5F5")
ax1.bar(x + width/2, cov_5, width, label="5 sentences", color="#1565C0")
ax1.set_ylabel("Word Coverage")
ax1.set_title("Summary Word Coverage")
ax1.set_xticks(x)
ax1.set_xticklabels(conv_names, rotation=15, ha="right")
ax1.legend()
ax1.grid(True, alpha=0.3, axis="y")

ax2.bar(x - width/2, comp_3, width, label="3 sentences", color="#66BB6A")
ax2.bar(x + width/2, comp_5, width, label="5 sentences", color="#2E7D32")
ax2.set_ylabel("Compression Ratio")
ax2.set_title("Summary Compression Ratio")
ax2.set_xticks(x)
ax2.set_xticklabels(conv_names, rotation=15, ha="right")
ax2.legend()
ax2.grid(True, alpha=0.3, axis="y")

plt.suptitle("TextRank Summarization — Evaluation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Similarity Matrix Heatmap

Visualize how sentences relate to each other in the graph.

In [ ]:
# ============================================================
# Cell 10 — Similarity matrix heatmap
# ============================================================
test_conv_name = "Bug Discussion"
test_msgs = [m for m in conversations[test_conv_name] if len(m.split()) > 2]

# Build raw (unnormalized) similarity matrix
n = len(test_msgs)
raw_sim = np.zeros((n, n))
tokenized = [TextRankSummarizer._tokenize(s) for s in test_msgs]
for i in range(n):
    for j in range(n):
        if i != j:
            raw_sim[i][j] = summarizer._cosine_similarity(tokenized[i], tokenized[j])

labels = [m[:35] + "..." if len(m) > 35 else m for m in test_msgs]

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(raw_sim, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(labels, fontsize=8)
ax.set_title(f"Cosine Similarity Matrix — {test_conv_name}", fontsize=12, fontweight="bold")

for i in range(n):
    for j in range(n):
        val = raw_sim[i][j]
        if val > 0.01:
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=7, color="white" if val > 0.3 else "black")

plt.colorbar(im)
plt.tight_layout()
plt.show()

## 8. Convergence Analysis

Track how PageRank scores converge over iterations.

In [ ]:
# ============================================================
# Cell 11 — PageRank convergence
# ============================================================
test_msgs = [m for m in conversations["Project Planning"] if len(m.split()) > 2]
sim_matrix = summarizer._build_similarity_matrix(test_msgs)

n = sim_matrix.shape[0]
scores = np.ones(n) / n
score_history = [scores.copy()]
diff_history = []

for iteration in range(50):
    new_scores = (1 - 0.85) / n + 0.85 * sim_matrix.T @ scores
    diff = np.abs(new_scores - scores).sum()
    diff_history.append(diff)
    scores = new_scores
    score_history.append(scores.copy())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Convergence (total difference per iteration)
ax1.plot(diff_history, color="#E53935", linewidth=2)
ax1.axhline(y=1e-5, color="gray", linestyle="--", alpha=0.5, label="Threshold")
converged_at = next((i for i, d in enumerate(diff_history) if d < 1e-5), len(diff_history))
ax1.axvline(x=converged_at, color="green", linestyle="--", alpha=0.5, label=f"Converged at {converged_at}")
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Total Score Difference")
ax1.set_title("PageRank Convergence")
ax1.set_yscale("log")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Score evolution per sentence
score_array = np.array(score_history)
for i in range(n):
    label = test_msgs[i][:30] + "..." if len(test_msgs[i]) > 30 else test_msgs[i]
    ax2.plot(score_array[:, i], label=label, linewidth=1.5)
ax2.set_xlabel("Iteration")
ax2.set_ylabel("Score")
ax2.set_title("Sentence Score Evolution")
ax2.legend(fontsize=6, loc="center left", bbox_to_anchor=(1, 0.5))
ax2.grid(True, alpha=0.3)

plt.suptitle("PageRank Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Converged after {converged_at} iterations (threshold = 1e-5)")

## 9. End-to-End Summarization Pipeline

Matches the `SummarizationPredictor` class from `ai/summarization/inference/predict.py`.

In [ ]:
# ============================================================
# Cell 12 — SummarizationPredictor (mirrors ai/summarization/inference/predict.py)
# ============================================================
class SummarizationPredictor:
    """Summarizes chat conversations using TextRank."""

    def __init__(self):
        self.summarizer = TextRankSummarizer()

    def predict(self, messages, num_sentences=5):
        if not messages:
            return "No messages to summarize."

        meaningful = [m for m in messages if len(m.split()) > 2]
        if not meaningful:
            return " ".join(messages[:num_sentences])

        return self.summarizer.summarize(meaningful, num_sentences=num_sentences)


# Test the full pipeline
predictor = SummarizationPredictor()

print("=" * 70)
print("End-to-End Summarization Pipeline")
print("=" * 70)

for name, messages in conversations.items():
    summary = predictor.predict(messages, num_sentences=3)
    print(f"\n[{name}]")
    print(f"  Input:   {len(messages)} messages")
    print(f"  Summary: {summary}")

# Edge cases
print("\n" + "=" * 70)
print("Edge Cases")
print("=" * 70)
print(f"  Empty:        {predictor.predict([])}")
print(f"  Short msgs:   {predictor.predict(['hi', 'ok', 'bye'])}")
print(f"  Single msg:   {predictor.predict(['This is the only message in the conversation.'])}")

## Summary

**TextRank** is fully unsupervised — no training data, no GPU, no model files needed.

| Aspect | Detail |
|---|---|
| Algorithm | PageRank on cosine-similarity sentence graph |
| Training | None (unsupervised) |
| Damping Factor | 0.85 (default) |
| Convergence | Typically < 20 iterations |
| Export Files | None needed — algorithm runs at inference time |

The `SummarizationPredictor` in `ai/summarization/inference/predict.py` uses this exact algorithm at runtime.